# Step 1: Setup & Understanding the Data

**Goal of this project:** Analyze one year of transaction data from a UK-based online retailer to answer business questions like: *Which product categories and regions drive the most revenue? Is there seasonality? Which segments underperform?*

**What is EDA?** EDA stands for *Exploratory Data Analysis* — before answering any business question, you first need to understand what's actually in your data: how many rows, which columns, what's missing, what looks broken. Skipping this step is the #1 reason analyses produce wrong conclusions.

**Dataset:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail) — 541,909 real transactions from a UK-based online gift retailer, Dec 2010 to Dec 2011. Public domain, free to use.

In [1]:
import pandas as pd

df = pd.read_excel("data/online_retail.xlsx")
df.shape

(541909, 8)

~542,000 rows, 8 columns. Let's look at the columns and a sample of rows.

In [2]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


**What each column means:**

| Column | Meaning |
|---|---|
| `InvoiceNo` | Order/receipt number. If it starts with a `C`, the row is a **cancellation**. |
| `StockCode` | Product ID |
| `Description` | Product name |
| `Quantity` | Units sold (negative = returned/cancelled) |
| `InvoiceDate` | Date and time of the transaction |
| `UnitPrice` | Price per unit, in GBP |
| `CustomerID` | Customer identifier (some transactions have none — e.g. guest checkouts) |
| `Country` | Customer's country |

There is no `Category` column and no cost/margin data in this dataset — so this analysis will focus on **revenue** (`Quantity x UnitPrice`), not margin. That's an honest limitation worth stating up front rather than inventing numbers that aren't there.

## Missing values

Real-world data is never perfect. Let's check which columns have gaps.

In [4]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

- `CustomerID` is missing in ~135,000 rows (25%) — likely guest checkouts without an account. We can't attribute these to a specific customer, but the transaction itself is still valid for revenue analysis.
- `Description` is missing in 1,454 rows — small enough to drop or investigate later.

We won't fix any of this yet — that's the job of Step 2 (Cleaning). Right now we're just taking stock.

## Time range and geography

In [5]:
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print("Countries:", df["Country"].nunique())
df["Country"].value_counts().head(10)

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Countries: 38


Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

One full year of data (Dec 2010 - Dec 2011) — good, that means we can look at seasonality without gaps. 38 countries, but the UK dominates (~91% of rows). Any country comparison will need to keep this imbalance in mind.

## Data quality issues to handle in Step 2

A few things stand out that we'll need to deal with before analyzing:

In [6]:
n_cancellations = df["InvoiceNo"].astype(str).str.startswith("C").sum()
n_negative_qty = (df["Quantity"] <= 0).sum()
n_bad_price = (df["UnitPrice"] <= 0).sum()
n_duplicates = df.duplicated().sum()

print(f"Cancellation invoices (InvoiceNo starts with 'C'): {n_cancellations}")
print(f"Rows with Quantity <= 0: {n_negative_qty}")
print(f"Rows with UnitPrice <= 0: {n_bad_price}")
print(f"Exact duplicate rows: {n_duplicates}")

Cancellation invoices (InvoiceNo starts with 'C'): 9288
Rows with Quantity <= 0: 10624
Rows with UnitPrice <= 0: 2517
Exact duplicate rows: 5268


**Summary of what Step 2 (Cleaning) needs to address:**
1. ~9,300 cancellation transactions (negative quantities) — decide whether to exclude them from revenue totals.
2. ~2,500 rows with a price of 0 or less — likely free samples or data errors, need investigation.
3. ~5,300 exact duplicate rows — probably scanning/logging errors, should be removed.
4. Missing `CustomerID` and `Description` values — decide what to do per analysis.

None of this data was invented — every number above comes directly from running `.sum()` on the real dataset above.